In [ ]:
import os

def find_dir(root, target_name):
    """Busca target_name como subcarpeta en cualquier nivel bajo root."""
    for dirpath, dirnames, _ in os.walk(root):
        if target_name in dirnames:
            return os.path.join(dirpath, target_name)
    return None

# El layout de /kaggle/input varía según la cuenta (a veces es
# /kaggle/input/<slug>/..., otras /kaggle/input/datasets/<slug>/...),
# así que buscamos la carpeta 'kaggle_input' sin asumir en qué nivel está.
INPUT_DIR = find_dir('/kaggle/input', 'kaggle_input')

if INPUT_DIR is None:
    print('No se encontró kaggle_input/. Árbol real de /kaggle/input:')
    for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
        print(dirpath, '->', dirnames, filenames[:5])
    raise FileNotFoundError("Revisa el árbol impreso arriba y ajusta target_name si tu carpeta se llama distinto.")

print('INPUT_DIR detectado:', INPUT_DIR)
print('Contenido:', os.listdir(INPUT_DIR))

## 1. Instalar dependencias
`miclustering` no está en PyPI, así que se instala directo desde GitHub. Ajusta si tu `requirements.txt` trae más paquetes pineados.

In [ ]:
req_path = f'{INPUT_DIR}/requirements.txt'
if os.path.isfile(req_path):
    !pip install -q -r {req_path}

# Reinstalamos miclustering siempre desde main de GitHub, sin caché, para
# garantizar que se usa el código con las distancias vectorizadas
# (BagCentroidCache) aunque requirements.txt traiga una versión anterior pineada.
!pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir \
    "miclustering @ git+https://github.com/Andmo2004/MIClustering.git"
!pip install -q optuna scikit-learn scipy pandas numpy

## 2. Preparar rutas
`/kaggle/input` es de solo lectura, así que la caché de matrices de distancia (que el script escribe) tiene que vivir en `/kaggle/working`. Si ya subiste una caché precomputada, la copiamos primero para no recalcularla.

In [ ]:
DATA_DIR = f'{INPUT_DIR}/data'
SCRIPT_PATH = f'{INPUT_DIR}/optimize_geometry.py'
OUTPUT_DIR = '/kaggle/working/resultados_geometria'
CACHE_DIR = '/kaggle/working/cache'

os.makedirs(CACHE_DIR, exist_ok=True)
src_cache = f'{INPUT_DIR}/cache'
if os.path.isdir(src_cache):
    !cp -n {src_cache}/*.npy {CACHE_DIR}/ 2>/dev/null || true

print('Datasets .arff detectados:')
!ls {DATA_DIR}

## 3. Smoke test cronometrado
Corre un solo estudio (1 dataset, 1 escalador, 1 distancia, 1 modelo) con pocos trials, mide cuánto tarda por trial y extrapola (muy aproximado) al grid completo.

**Nota sobre modelos:**
- `midbscan`, `mikmedoids` y `cosmic` utilizan directamente la matriz de distancias precomputada (`precomputed_matrix`).
- `mikmeans` recalcula centroides sintéticos dinámicos en cada iteración, pero ahora utiliza **distancias vectorizadas BLAS (N×K)** con **`BagCentroidCache` precomputada** una única vez por combinación (dataset, escalador, distancia). Corre en pocos milisegundos por trial (~17.5x más rápido), por lo que puedes medir con `mikmeans` o `mikmedoids` indistintamente.
- Se dispone de los parámetros `--max-iters` y `--tol` para controlar el número de iteraciones y tolerancia de convergencia de los modelos iterativos.

In [ ]:
import subprocess, time

DEVICE = 'cpu'  # con matrices ya cacheadas, cpu basta; cuda solo importaría si faltara alguna matriz por calcular

SMOKE_DATASET = 'musk1'
SMOKE_SCALER = 'MinMaxScaler'
SMOKE_DISTANCE = 'cauchy_schwarz'
SMOKE_MODEL = 'mikmeans'  # 'mikmeans' ahora usa distancias vectorizadas (N×K) y BagCentroidCache; 'mikmedoids' usa precomputed_matrix
SMOKE_TRIALS = 5
MAX_ITERS = 100
TOL = 0.01

t0 = time.perf_counter()
subprocess.run([
    'python', SCRIPT_PATH,
    '--data-dir', DATA_DIR,
    '--output-dir', f'{OUTPUT_DIR}_smoke',
    '--cache-dir', CACHE_DIR,
    '--datasets', SMOKE_DATASET,
    '--scalers', SMOKE_SCALER,
    '--distances', SMOKE_DISTANCE,
    '--models', SMOKE_MODEL,
    '--n-trials', str(SMOKE_TRIALS),
    '--patience', str(SMOKE_TRIALS),
    '--max-iters', str(MAX_ITERS),
    '--tol', str(TOL),
    '--device', DEVICE,
], check=True)
elapsed = time.perf_counter() - t0

# Con --scalers/--distances fijos a un solo valor, este SMOKE_TRIALS ya
# corresponde 1:1 al número real de trials ejecutados.
sec_per_trial = elapsed / SMOKE_TRIALS
print(f'\nSmoke test: {elapsed:.1f}s para {SMOKE_TRIALS} trials -> {sec_per_trial:.2f} s/trial ({SMOKE_DATASET}/{SMOKE_SCALER}/{SMOKE_DISTANCE}/{SMOKE_MODEL})')

# Extrapolación al grid completo por defecto.
# Con MIKMeans vectorizado y matrices cacheadas, todos los modelos corren en milisegundos por trial.
n_datasets, n_scalers, n_distances, n_models, n_trials = 10, 2, 6, 4, 50
total_trials = n_datasets * n_scalers * n_distances * n_models * n_trials
est_hours = (total_trials * sec_per_trial) / 3600
print(f'Grid completo por defecto: {total_trials} trials -> ~{est_hours:.1f} h estimadas (piso aproximado)')

## 4. Optimizar: paralelizar por dataset

`study.optimize()` está fijado en el script a `n_jobs=1` (un trial a la vez dentro de cada estudio), y `--n-jobs` del CLI solo afecta al cómputo de la matriz de distancias, no a los trials. Pero los 480 estudios son independientes entre sí (cada uno es su propia combinación dataset/escalador/distancia/modelo), así que en vez de paralelismo *dentro* de un estudio, conviene paralelismo *entre* estudios: lanzar varios procesos de `optimize_geometry.py`, cada uno acotado a un solo `--datasets`, corriendo al mismo tiempo — uno por núcleo disponible. Como cada proceso solo toca los archivos de caché de su propio dataset, no hay condiciones de carrera.

Esto no toca el script ni requiere tocar la librería: es paralelismo a nivel de proceso, usando el mismo CLI de siempre.

In [ ]:
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed

print('CPUs disponibles:', os.cpu_count())

N_TRIALS = 250         # Presupuesto para mikmeans y mikmedoids
PATIENCE = 50          # Early stopping para mikmeans y mikmedoids
DENSITY_N_TRIALS = 50  # Presupuesto (max_iter) para midbscan y cosmic
DENSITY_PATIENCE = 30  # Early stopping para midbscan y cosmic
MAX_ITERS = 100
TOL = 0.01

# Ejecución completa: los 4 modelos se optimizan sobre todos los datasets del directorio
RUN_MODELS = ['midbscan', 'mikmeans', 'mikmedoids', 'cosmic']

PAR_OUTPUT_DIR = f'{OUTPUT_DIR}_par'

ALL_DATASETS = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(DATA_DIR)
    if f.endswith('.arff')
)

print('Datasets a repartir:', ALL_DATASETS)
print('Modelos a optimizar:', RUN_MODELS)


def run_for_dataset(dataset_name):
    out_dir = f'{PAR_OUTPUT_DIR}/{dataset_name}'

    result = subprocess.run([
        'python', SCRIPT_PATH,
        '--data-dir', DATA_DIR,
        '--output-dir', out_dir,
        '--cache-dir', CACHE_DIR,
        '--datasets', dataset_name,
        '--models', *RUN_MODELS,
        '--n-trials', str(N_TRIALS),
        '--patience', str(PATIENCE),
        '--density-n-trials', str(DENSITY_N_TRIALS),
        '--density-patience', str(DENSITY_PATIENCE),
        '--max-iters', str(MAX_ITERS),
        '--tol', str(TOL),
        '--seed', '42',
        '--device', DEVICE,
    ], capture_output=True, text=True)

    return dataset_name, result.returncode, result.stderr[-800:]


MAX_WORKERS = min(
    os.cpu_count() or 4,
    len(ALL_DATASETS)
)

print(f'Lanzando con {MAX_WORKERS} procesos en paralelo...')

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {
        ex.submit(run_for_dataset, d): d
        for d in ALL_DATASETS
    }

    for fut in as_completed(futures):
        d, code, err_tail = fut.result()

        print(
            f'[{d}] '
            f'{"OK" if code == 0 else f"ERROR (code {code})"}'
        )

        if code != 0:
            print(err_tail)

Cada proceso escribe su propio `resultados_optimizacion_geometria.csv` dentro de `{PAR_OUTPUT_DIR}/<dataset>/` (el script sobrescribe ese archivo por invocación, no lo acumula entre corridas — ver sección 5). Une los resultados así:

In [ ]:
import pandas as pd

csvs = glob.glob(f'{PAR_OUTPUT_DIR}/*/resultados_optimizacion_geometria.csv')
combined = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
combined_path = f'{PAR_OUTPUT_DIR}/resultados_combinados.csv'
combined.to_csv(combined_path, index=False)
print(f'{len(combined)} filas combinadas de {len(csvs)} datasets -> {combined_path}')

## 4b. Alternativa secuencial (más simple, sin paralelizar)

Si prefieres una sola invocación sin paralelismo (más simple de depurar, pero sin el speedup de la sección 4):

Dos cosas a tener presentes si la corres así:
- La tabla maestra consolidada (`resultados_optimizacion_geometria.csv`, las pivot tables y `mejores_configuraciones.csv`) solo se escribe **al final de todo el pipeline**. Si la sesión se corta antes, la pierdes. Lo que **sí** sobrevive porque se guarda estudio por estudio: los CSV de `trials/<dataset>/<modelo>/...` y la caché `.npy`. Si te corta a mitad de camino, no perdiste los datos crudos, solo el resumen — se puede reconstruir después concatenando esos CSV.
- Lánzala con **Save Version -> Save & Run All (Commit)** en vez de dejarla corriendo en modo interactivo: corre en segundo plano y no depende de que mantengas la pestaña del navegador abierta.

Si tu extrapolación del smoke test (con este nuevo N_TRIALS) dio varias veces el límite de sesión, mejor trocear explícitamente con `--datasets` / `--models` en 2+ runs en vez de arriesgar la tabla maestra.

In [ ]:
N_TRIALS = 250         # Presupuesto para mikmeans y mikmedoids
PATIENCE = 50          # Early stopping para mikmeans y mikmedoids
DENSITY_N_TRIALS = 50  # Presupuesto (max_iter) para midbscan y cosmic
DENSITY_PATIENCE = 30  # Early stopping para midbscan y cosmic
MAX_ITERS = 100
TOL = 0.01

# Run completo: sin --datasets, --scalers, --distances ni --models
# procesa todo lo que haya en DATA_DIR con todos los escaladores/distancias/modelos por defecto.
!python {SCRIPT_PATH} \
  --data-dir {DATA_DIR} \
  --output-dir {OUTPUT_DIR} \
  --cache-dir {CACHE_DIR} \
  --n-trials {N_TRIALS} \
  --patience {PATIENCE} \
  --density-n-trials {DENSITY_N_TRIALS} \
  --density-patience {DENSITY_PATIENCE} \
  --max-iters {MAX_ITERS} \
  --tol {TOL} \
  --seed 42 \
  --n-jobs -1 \
  --device {DEVICE}

## 5. Empaquetar resultados para persistirlos entre sesiones

In [ ]:
!zip -qr /kaggle/working/resultados_geometria.zip {OUTPUT_DIR} 2>/dev/null || true
!zip -qr /kaggle/working/resultados_geometria_par.zip {OUTPUT_DIR}_par 2>/dev/null || true
!zip -qr /kaggle/working/cache.zip {CACHE_DIR}
print('Listo: zips en /kaggle/working (el que corresponda según qué sección usaste: 4 o 4b)')